<a href="https://colab.research.google.com/github/BBotond03/MTDK_2026/blob/main/Visualisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1. Bemeneti adatok: (x, y, z,  vx, vy, vz)
# Az első 3 szám a középpont (pozíció).
# A második 3 szám az irányvektor. Mivel az YZ síkban vagyunk, a vx végig 0.
utvonal = [
    (0, 0, 0,    0, 1, 0),    # Y felé néz
    (1, 1, 0.5,  0, 1, 1),    # 45 fokban Y és Z között
    (2, 3, 1,    0, 0, 1),    # Egyenesen felfelé (Z) néz
    (4, 2, 1.5,  0, -1, 1),   # Z és -Y felé dől
    (5, 0, 1,    0, -1, 0),   # -Y felé néz
    (3, -2, 0.5, 0, -1, -1)   # Lefelé és hátra dől
]

sugar = 1.0
magassag = 0.2
felbontas = 20

theta = np.linspace(0, 2*np.pi, felbontas)
z_space = np.linspace(-magassag/2, magassag/2, 2)
theta_grid, z_grid = np.meshgrid(theta, z_space)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

def update(frame):
    ax.clear()

    # Adatok kipakolása
    cx, cy, cz, vx, vy, vz = utvonal[frame]

    # 1. LÉPÉS: A kapott irányvektor normalizálása (hogy a hossza pontosan 1 legyen)
    n = np.array([vx, vy, vz], dtype=float)
    n /= np.linalg.norm(n)

    # 2. LÉPÉS: Keresünk két egymásra merőleges vektort (v1, v2), amik kifeszítik a kört.
    # Mivel a te vektorod az YZ síkban van, az X-tengely (1, 0, 0) garantáltan merőleges rá.
    u = np.array([1.0, 0.0, 0.0])

    # Biztonsági ellenőrzés: ha később mégis az X tengely mentén adnál meg vektort
    if abs(n[0]) > 0.9:
        u = np.array([0.0, 1.0, 0.0])

    v1 = np.cross(n, u)
    v1 /= np.linalg.norm(v1)
    v2 = np.cross(n, v1)

    # 3. LÉPÉS: Hálópontok kiszámítása a vektorok alapján (Lineáris kombináció)
    # Ez a rész automatikusan forgatja és dönti a hengert a térben
    X = cx + sugar * np.cos(theta_grid) * v1[0] + sugar * np.sin(theta_grid) * v2[0] + z_grid * n[0]
    Y = cy + sugar * np.cos(theta_grid) * v1[1] + sugar * np.sin(theta_grid) * v2[1] + z_grid * n[1]
    Z = cz + sugar * np.cos(theta_grid) * v1[2] + sugar * np.sin(theta_grid) * v2[2] + z_grid * n[2]

    # Kirajzolás
    ax.plot_surface(X, Y, Z, color='cyan', alpha=0.8, edgecolor='none')

    # Tengelyek fixálása, hogy a tér ne mozogjon a tárggyal együtt
    ax.set_xlim(-1, 6)
    ax.set_ylim(-3, 4)
    ax.set_zlim(-1, 3)
    ax.set_xlabel('X tengely')
    ax.set_ylabel('Y tengely')
    ax.set_zlabel('Z tengely')
    ax.set_title(f"Pozíció: ({cx}, {cy}, {cz})\nVektor: ({vx}, {vy}, {vz})")

# Animáció összerakása
ani = FuncAnimation(fig, update, frames=len(utvonal), interval=1000)

plt.close() # Ne duplikálja a képet Colabban

# Videó beillesztése a cella alá
HTML(ani.to_html5_video())

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1. Bemeneti adatok: (cx, cy, cz,  vx, vy, vz)
# Most az irányvektor végig (1, 0, 0), tehát a korong az YZ síkkal párhuzamosan áll.
utvonal = [
    (0, 0, 0,    1, 0, 0),
    (1, 1, 0.5,  1, 0, 0),
    (2, 2, 1.5,  1, 0, 0),
    (3, 0, 2.0,  1, 0, 0),
    (4, -2, 1.0, 1, 0, 0),
    (5, -1, 0.0, 1, 0, 0)
]

sugar_max = 1.0
felbontas_kerek = 30
felbontas_sugar = 10

r = np.linspace(0, sugar_max, felbontas_sugar)
theta = np.linspace(0, 2*np.pi, felbontas_kerek)
r_grid, theta_grid = np.meshgrid(r, theta)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

def update(frame):
    ax.clear()

    # Adatok kipakolása
    cx, cy, cz, vx, vy, vz = utvonal[frame]

    # Vektor normalizálása
    n = np.array([vx, vy, vz], dtype=float)
    n /= np.linalg.norm(n)

    # Bázisvektorok a korong síkjához
    u = np.array([1.0, 0.0, 0.0])

    # Ha a normálvektor (majdnem) teljesen egybeesik az X tengellyel,
    # akkor u-t Y-ra cseréljük, hogy a vektoriális szorzat (cross) működjön!
    if abs(n[0]) > 0.9:
        u = np.array([0.0, 1.0, 0.0])

    v1 = np.cross(n, u)
    v1 /= np.linalg.norm(v1)
    v2 = np.cross(n, v1)

    # Tömör korong pontjainak kiszámítása
    X = cx + r_grid * np.cos(theta_grid) * v1[0] + r_grid * np.sin(theta_grid) * v2[0]
    Y = cy + r_grid * np.cos(theta_grid) * v1[1] + r_grid * np.sin(theta_grid) * v2[1]
    Z = cz + r_grid * np.cos(theta_grid) * v1[2] + r_grid * np.sin(theta_grid) * v2[2]

    ax.plot_surface(X, Y, Z, color='orange', alpha=0.9, edgecolor='none')

    # Tengelyek rögzítése
    ax.set_xlim(-1, 6)
    ax.set_ylim(-3, 3)
    ax.set_zlim(-1, 3)
    ax.set_xlabel('X tengely')
    ax.set_ylabel('Y tengely')
    ax.set_zlabel('Z tengely')
    ax.set_title(f"Pozíció: ({cx}, {cy}, {cz})\nIrányvektor: ({vx}, {vy}, {vz})")

# Animáció összerakása
ani = FuncAnimation(fig, update, frames=len(utvonal), interval=1000)

plt.close()

# Videó legenerálása
HTML(ani.to_html5_video())